<a href="https://colab.research.google.com/github/emgrimm42/NLP-Hausarbeit/blob/main/analyse_4_synonyme2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import numpy as np
from wordcloud import WordCloud
import plotly.graph_objects as go

Unser vorheriger Schritt hat ergeben dass fasst alle unserer bisherigen Synonyme gar nicht im Text vorkamen und tatsächlich fast nur das sehr allgemeine Wort Lüge benutzt wurde. Um zu verhindern, dass Synonyme übersehen werden sollten wir zunächst schauen ob es nicht vielleicht doch noch ein paar Synonyme gibt die wir vergessen haben. Dafür lassen wir eine Semantic search nach similar words zu unserer ursprünglichen Lügenpresse-Synonyme Liste suchen.
Das machen wir auf dem Volltext excel dokument um sicherzustellen dass wir auch nichts übersehen.

In [ ]:
# Replace 'your_excel_file.xlsx' with the actual path to your Excel file
df = pd.read_excel('/content/Volltext_2751569-2.xlsx')

# Now you can work with the DataFrame 'df'
df.head()

In [ ]:
# @markdown #### Installing the Sentence Tranfromers from HuggingFace
!pip install sentence-transformers

Probelauf nur mit Lügenpresse

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re
from sentence_transformers import SentenceTransformer
from collections import Counter

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zäöüß\s]', '', text)
    return text

def get_unique_words(text):
    words = text.split()
    return list(set(words))

def find_similar_words(df, target_word, model, top_n=40):
    # Preprocess the text
    df['processed_text'] = df['plainpagefulltext'].apply(preprocess_text)

    # Get unique words from all texts
    all_words = []
    for text in df['processed_text']:
        all_words.extend(get_unique_words(text))

    # Get unique words and their frequencies
    word_freq = Counter(all_words)
    unique_words = list(word_freq.keys())

    print(f"Number of unique words: {len(unique_words)}")

    # Encode the target word and unique words
    target_embedding = model.encode([target_word])
    word_embeddings = model.encode(unique_words)

    # Calculate similarities
    similarities = cosine_similarity(target_embedding, word_embeddings)[0]

    # Create a DataFrame with words and their similarities
    word_sim_df = pd.DataFrame({
        'word': unique_words,
        'similarity': similarities
    })

    # Sort by similarity and get top N results
    top_similar = word_sim_df.sort_values('similarity', ascending=False).head(top_n)

    return top_similar

# Load the pre-trained multilingual model
print("Loading the sentence transformer model...")
model = SentenceTransformer('sentence-transformers/LaBSE')
print("Model loaded successfully.")

target_word = "Lügenpresse"

print(f"\nFinding words similar to '{target_word}'...")
similar_words = find_similar_words(df, target_word, model)

print("\nMost similar words:")
print(similar_words)

Ergebnis:
Most similar words:
                   word  similarity
78522      lügentrafens    0.704311
197172      lügentrafen    0.679021
142663       lügenfluth    0.676267
171901         lügenpot    0.654946
151449   lügengechichte    0.636898
119287      lügenkaiers    0.621344
121332     leipzigdruck    0.620508
87265        lügenkünte    0.620120
81895         lügentoff    0.619006
140610        lügengeit    0.613418
29562         lügenytem    0.613018
218118       lügenmaske    0.612543
254591       lügenteige    0.598347
252651   lügenberichten    0.597791
204649     lügengepinnt    0.595968
200279    lügenberichte    0.595535
256833      lügentaktik    0.588145
17415            presse    0.585795
219299       lügenboten    0.581949
261009      lügenfabrik    0.580486
135309     lügenblätter    0.577120
68484       lügengewebe    0.573101
115214      leuenprozeß    0.572053
125755      druckenlaen    0.571873
102845     leepublicums    0.569173
234590        lügenpiel    0.558390
152264      leepublikum    0.554755
32003       jurazeitung    0.552914
106462    ausdrucksweie    0.550770
58267         elzeitung    0.549673
35883   lügenhaftigkeit    0.549310
209426     lügenhaftete    0.547989
219292    lügenregiment    0.546952
14291              lüge    0.545031
5369              lügen    0.541098
224519        lenprozee    0.539667
65573           plagiat    0.538747
84169       nachdrucker    0.535719
156805     polizeidruck    0.534673
150206       lenprozees    0.533451

Okay und jetzt noch mal mit der Liste an Synonymen.
Da wir nur nach ähnlichen und nicht nach identischen Wörtern suchen können wir uns auf eine Schreibweise jedes Wortes beschrenken.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re
from sentence_transformers import SentenceTransformer
from collections import Counter

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zäöüß\s]', '', text)
    return text

def get_unique_words(text):
    words = text.split()
    return list(set(words))

def find_similar_words(df_processed, target_word, model, top_n=40):
    # Get unique words from all texts
    all_words = []
    for text in df_processed['processed_text']:
        all_words.extend(get_unique_words(text))

    # Get unique words and their frequencies
    word_freq = Counter(all_words)
    unique_words = list(word_freq.keys())

    print(f"Number of unique words: {len(unique_words)}")

    # Encode the target word and unique words
    target_embedding = model.encode([target_word])
    word_embeddings = model.encode(unique_words)

    # Calculate similarities
    similarities = cosine_similarity(target_embedding, word_embeddings)[0]

    # Create a DataFrame with words and their similarities
    word_sim_df = pd.DataFrame({
        'word': unique_words,
        'similarity': similarities
    })

    # Sort by similarity and get top N results
    top_similar = word_sim_df.sort_values('similarity', ascending=False).head(top_n)

    return top_similar

# Load the pre-trained multilingual model
print("Loading the sentence transformer model...")
model = SentenceTransformer('sentence-transformers/LaBSE')
print("Model loaded successfully.")

# Preprocess the text once outside the loop for efficiency
df['processed_text'] = df['plainpagefulltext'].apply(preprocess_text)

# Define the list of target words
target_words_list = [
    "Lüge",
    "Falschmeldung",
    "Lügenblatt",
    "Hetzblatt",
    "Hetzpresse",
    "Schmierblatt",
    "Schmierpresse",
    "Pamphletpresse",
    "Parteipresse",
    "Demagogenpresse",
    "Revolutionspresse",
    "radikale Presse",
    "ultraradikale Presse",
    "jüdische Presse",
    "Judenpresse",
    "liberale Presse",
    "Freimaurerpresse",
    "Logenpresse",
    "Pressunfug",
    "Wühlerpresse",
    "Gegenpresse"
]

# Loop through each target word and find similar words
for target_word_item in target_words_list:
    print(f"\nFinding words similar to '{target_word_item}'...")
    similar_words = find_similar_words(df, target_word_item, model)
    print(f"Most similar words for '{target_word_item}':")
    print(similar_words)


Ergebnisse:
Finding words similar to 'Lüge'...
Number of unique words: 269204
Most similar words for 'Lüge':
                word  similarity
5369           lügen    0.870745
14291           lüge    0.866744
5913             lie    0.855874
1928            lies    0.822375
140610     lügengeit    0.804888
238626         lagau    0.783272
140020          liar    0.782557
180633       mentirt    0.781584
13293        gelogen    0.776908
165983      mentirte    0.776453
209426  lügenhaftete    0.776386
63155       mentiren    0.774816
29562      lügenytem    0.773420
60298           lieg    0.767382
27475           lied    0.766673
266145         liehe    0.764282
47048    lügenhaften    0.758576
74273     lügenhafte    0.757297
254591    lügenteige    0.750761
123139     lügenhaft    0.749754
125208        fausse    0.748564
210265          liee    0.747904
129109          lügt    0.746228
75171        fausses    0.741804
1492           liege    0.741608
95773         lapsus    0.739893
110967    liegenheit    0.739801
42214         gelobe    0.735004
171901      lügenpot    0.734242
19945           lieh    0.733713
226879         liari    0.732840
31188         liarde    0.731714
79853           liel    0.728667
142663    lügenfluth    0.726292
89588          lieds    0.725913
64137          gelob    0.724671
130202  liegenheiten    0.721061
243931        falsos    0.719800
81895      lügentoff    0.718329
197172   lügentrafen    0.717298

Finding words similar to 'Falschmeldung'...
Number of unique words: 269204
Most similar words for 'Falschmeldung':
                           word  similarity
237917                   erreur    0.705799
23473                      feil    0.702164
254591               lügenteige    0.701123
244677                    error    0.698125
251490              feilbietung    0.696070
71131            feldauftellung    0.688392
192652                  feilbot    0.683083
224939                fehlerhaf    0.671510
239206             rechenfehler    0.669523
159115          rechnungsfehler    0.663950
218355          terlaungsfehler    0.652054
252651           lügenberichten    0.651902
187496  druckfehlerberichtigung    0.650490
167149       uebereilungsfehler    0.646450
40831               feilgeboten    0.645824
250098                rangtitel    0.644355
194144                fehlchluß    0.644059
39825                formfehler    0.640425
218118               lügenmaske    0.639731
200279            lügenberichte    0.638619
207300          feilzubietenden    0.638344
104894             fehlerhaftes    0.632465
41760                  feilheit    0.630218
207285               feilbieten    0.630101
75859           gechäftsbericht    0.629634
261190             feilzubieten    0.627098
87447                     errat    0.626187
140610                lügengeit    0.624819
244722                incorrect    0.623964
119153                feilboten    0.623042
109059                 urfehler    0.622126
106904                  fehlern    0.620730
7102                     fehler    0.613500
29562                 lügenytem    0.612985
111454         nichtigerklärung    0.612536
23107                 fehlchlug    0.612360
66399              fehlerhaften    0.611729
82868     nichtigkeitserklärung    0.609812
131896                rangtreit    0.609079
177512          falliterklärung    0.608905

Finding words similar to 'Lügenblatt'...
Number of unique words: 269204
Most similar words for 'Lügenblatt':
                     word  similarity
135309       lügenblätter    0.764393
171901           lügenpot    0.736691
151449     lügengechichte    0.727316
83495        jeuitenblatt    0.712641
218118         lügenmaske    0.703653
142663         lügenfluth    0.699376
243608        feigenblatt    0.699053
115683     erbauungsblatt    0.698466
92633       gierungsblatt    0.690664
78522        lügentrafens    0.689441
197172        lügentrafen    0.688383
81895           lügentoff    0.682086
161950         tungsblatt    0.680083
29562           lügenytem    0.671988
140610          lügengeit    0.671204
109993           regblatt    0.670827
131552      jeuitenblatte    0.667489
61879     oppoitionsblatt    0.662055
178750       verordnblatt    0.660514
26901           flugblatt    0.659707
122765          llandbuch    0.657516
87265          lügenkünte    0.656093
254591         lügenteige    0.655410
196363         börenblatt    0.654211
202935           lebrbuch    0.648555
219299         lügenboten    0.644396
119287        lügenkaiers    0.643887
177875         blaebälgen    0.642670
68484         lügengewebe    0.642077
95175            lappland    0.641123
252832       feigenblatts    0.640501
128053  ragierungsblattes    0.640400
180192         späheblatt    0.640292
239013       verlagsbuchh    0.639108
80593             leebuch    0.638827
173747          kleeblatt    0.637017
15280      pondenzblattes    0.636073
38127            tagblatt    0.635731
37215          volksblatt    0.635622
220726     mailänderblatt    0.634129

Finding words similar to 'Hetzblatt'...
Number of unique words: 269204
Most similar words for 'Hetzblatt':
                  word  similarity
212650       herzblatt    0.759726
127464        zelblatt    0.670502
102124      geetzblatt    0.657793
74128         hatzfeld    0.653498
212540    holzdiebtahl    0.647419
141795         hatkett    0.644328
169712       zahlbrett    0.639759
13312        hatzfeldt    0.639725
88250       haspthebel    0.637348
124741       witzblatt    0.633341
46809         hofblatt    0.629106
137365     chaftsgelüt    0.627539
38910         herzblut    0.626097
83599         hochtift    0.624158
153543      denzblatte    0.622490
234867       genzblatt    0.621311
164579       höchtelbt    0.617432
206061       whigblatt    0.616459
144280      holzchnitt    0.616270
193634  hetzgeellchaft    0.616054
178822       chenblatt    0.615461
247668       hausbeitz    0.614186
216465       hanfbetel    0.613388
49300      holzchnitte    0.612544
105586     geetzblatte    0.612248
75723         hintrebt    0.610061
60756        herzblute    0.609304
193758      stichblatt    0.607990
266772       herbtluft    0.607893
158779      richtsbret    0.607482
257664        hiebspot    0.606416
80815           hetzte    0.604366
148881          hattte    0.604172
216489    pottchlechte    0.604094
145569        hiobspot    0.603900
154034         thibert    0.603899
211731       altbetche    0.603239
82148        herbtzeit    0.601600
235124       brethafte    0.601233
89716        tatthatte    0.601206

Finding words similar to 'Hetzpresse'...
Number of unique words: 269204
Most similar words for 'Hetzpresse':
                     word  similarity
234948        geitesdruck    0.702832
129285       geitesdrucks    0.693924
33067             abdruck    0.670325
3608            nachdruck    0.666584
223661  nachdrucksprocees    0.661670
121332       leipzigdruck    0.651210
649              ausdruck    0.646799
247634        händedrucks    0.645753
17415              presse    0.643797
266857            pressen    0.643270
67959         drückendten    0.639641
156842           pression    0.638974
232558              press    0.633574
68671        unterdrucken    0.633435
141776          streszewo    0.630805
219614             pressa    0.630013
26410            abdrucks    0.629763
35423          händedruck    0.629466
55506            ansdruck    0.627137
89832        gewiensdruck    0.623567
228770           zeidruck    0.622520
78689          widerdruck    0.622048
147169         gegendruck    0.621810
55697          ausdruckes    0.620212
43867         drückendter    0.616598
21161           ausdrucks    0.615680
106462      ausdrucksweie    0.615613
116580     geammtausdruck    0.615510
26377            abdrucke    0.614994
83255          nachdrucks    0.614581
68729       unterdruckten    0.610604
122458          abdruckes    0.610328
204503        händedrucke    0.609893
64816        gegendruckes    0.608901
196428         drücktheit    0.608204
265046       nachdruckame    0.606376
121183        nachdruckes    0.600027
156805       polizeidruck    0.599982
52658           abdrucken    0.599226
84169         nachdrucker    0.597454

Finding words similar to 'Schmierblatt'...
Number of unique words: 269204
Most similar words for 'Schmierblatt':
                       word  similarity
73552          schmeißblatt    0.798609
208333          scherzblatt    0.756294
193758           stichblatt    0.750767
64513           bundesblatt    0.742675
241158           adreßblatt    0.737839
82450          winckelblech    0.727353
236743           schlußheft    0.715876
61879       oppoitionsblatt    0.705257
196363           börenblatt    0.704939
6757          haltungsblatt    0.697944
220726       mailänderblatt    0.693242
243608          feigenblatt    0.690965
109597           kreisblatt    0.688804
49606            rungsblatt    0.688161
30594            hauptblatt    0.682558
180192           späheblatt    0.674402
26901             flugblatt    0.672086
242942           zeigeblatt    0.670990
201380          steuerlaten    0.668668
128053    ragierungsblattes    0.667837
185268          zifferblatt    0.667483
83495          jeuitenblatt    0.663983
195437  verkündigungsblatte    0.659937
115683       erbauungsblatt    0.659020
178750         verordnblatt    0.656914
258320          schandblatt    0.655143
136034           spaltzeile    0.652902
69928          potamtsblatt    0.648064
250633      ergänzungsblatt    0.647971
131552        jeuitenblatte    0.645221
133774    korrepondenzblatt    0.644255
92633         gierungsblatt    0.644105
41229              domblatt    0.643605
263793          seitenthüre    0.642895
112752      schaaunkelbrett    0.642689
243943             mitblatt    0.642317
100682          schmähblatt    0.641435
109993             regblatt    0.638817
55060            titelblatt    0.636966
152279      neuigkeitsblatt    0.636098

Finding words similar to 'Schmierpresse'...
Number of unique words: 269204
Most similar words for 'Schmierpresse':
                     word  similarity
137645      kupferdrucker    0.713601
49541        stempeltaren    0.690894
153652        stempeltare    0.676950
201683        preßmachine    0.662014
106462      ausdrucksweie    0.651786
149490       stempelteuer    0.646021
57722      präentirteller    0.645443
43867         drückendter    0.645031
84169         nachdrucker    0.641648
70091         steuerruder    0.632481
17415              presse    0.625829
266857            pressen    0.624185
186015        spießträger    0.622135
121332       leipzigdruck    0.620851
217200       steindrucker    0.615853
91987   brochürenchreiber    0.615495
201870         stönchrape    0.615252
206923     krämereibetrie    0.612926
35323       compressorium    0.612824
223661  nachdrucksprocees    0.612423
229341   rathsbuchdrucker    0.612374
179460        steuerdruck    0.611871
215923     preßverbrecher    0.610432
195035      dampfmaschine    0.609600
125991      schriftchauer    0.608519
122318    strumpfwirkerar    0.607599
217440    präventivmittel    0.607512
167343        majeimdruck    0.606970
76068        steuerdrucke    0.605718
193027       dampfmachine    0.604369
210424           drückter    0.604183
245616   kupferdruckvelin    0.602588
234948        geitesdruck    0.602060
98713            zöppritz    0.601541
160672     druckerlaubniß    0.600374
232558              press    0.597546
199164       struwelpeter    0.596343
184654   steuereintreiber    0.595581
201894       steuerruders    0.593898
76633         buchdrucker    0.593690

Finding words similar to 'Pamphletpresse'...
Number of unique words: 269204
Most similar words for 'Pamphletpresse':
                      word  similarity
84709         pamphletiten    0.710324
154582          pamphletit    0.699202
72056            pamphlete    0.691588
135161          pamphleten    0.684399
31665             pamphlet    0.682230
223796    pamphletchreiber    0.672879
90331            pamphlets    0.667192
185090             pamphle    0.663780
118306    druckvelinpapier    0.656969
209954         druckpapier    0.651432
151661       stempelpapier    0.650562
66938       druckpapiernen    0.644009
238813          pappelthor    0.635139
97838         papieremiion    0.627860
17415               presse    0.626810
122663       prachtabdruck    0.625586
157827       papiertreifen    0.623931
121777    pergamenttableau    0.618287
42729         papierfetzen    0.616421
156805        polizeidruck    0.615290
84169          nachdrucker    0.615267
70521           löchpapier    0.612284
40902         papiercheere    0.611718
236871      papiervorräthe    0.609863
214575        papiermühlen    0.609805
137645       kupferdrucker    0.609555
74591           foliomappe    0.608990
110320         papieredikt    0.608623
231586       polizeiplakat    0.605329
223552              stampa    0.605301
114628       löchpapierner    0.604840
223661   nachdrucksprocees    0.602801
125755         druckenlaen    0.602659
200235  papierdurchchoenes    0.600669
246540            stampfer    0.600405
164207       auschußpapier    0.599581
149947     stempelpatentes    0.594908
99190             papierme    0.594293
253764     pferdegetrappel    0.593185
236977        papiertempel    0.589768

Finding words similar to 'Parteipresse'...
Number of unique words: 269204
Most similar words for 'Parteipresse':
                      word  similarity
260041     parteinachricht    0.738277
95368        parteipolemik    0.737551
187069        parteiblatte    0.725324
221430         parteiblatt    0.719790
93386     parteireibereien    0.718901
111696        parteiabicht    0.717787
215069         parteiüchti    0.717519
183255      parteiauflöung    0.716719
254957        parteifetzug    0.715694
229321      parteifraktion    0.715428
107096       parteibechluß    0.715264
79728       parteiumtriebe    0.715260
97354            parteibaß    0.713669
81012        parteiblätter    0.713331
223313      parteigetreibe    0.712841
104960     parteihäuptling    0.711936
26035       partianenkorps    0.711066
155941          parteiblät    0.710776
201578        parteifahnen    0.704524
249384     partianenbanden    0.704275
238118          parteibuch    0.702944
204052     parteiausichten    0.701734
260916        parteigelüte    0.698821
127872    parteifraktionen    0.696900
128309         parteienlut    0.695553
117627       parteiophitik    0.695404
112001       parteimanifet    0.695381
91786        parteigezänks    0.693895
267204   parteibegüntigung    0.693726
102424      parteizeitblat    0.693701
91539        parteihäupter    0.693091
58386            parteifüh    0.692723
254960   parteienaufregung    0.691752
152440       parteipolitik    0.691698
151779  parteiverbitterung    0.691540
202609           parteifär    0.691398
203700          parteilüge    0.691098
81339    parteiverbienheit    0.690567
212949       parteieinfluß    0.690126
146892        parteichrift    0.690125

Finding words similar to 'Demagogenpresse'...
Number of unique words: 269204
Most similar words for 'Demagogenpresse':
                               word  similarity
256762            demagogendespotie    0.680000
196626           demagogenriecherei    0.663134
237897              demagogenkünten    0.662145
202356               demagogenkünte    0.655646
261617      demagogichdemokratichen    0.655008
175833            demagogenriechern    0.645757
123801          demagogenwirthchaft    0.641162
203817       demagogenunteruchungen    0.640170
260000          demagogenverfolgung    0.622809
230511                 demagogismus    0.622075
124058             demagogenlorbeer    0.617348
162589             demagogengechäft    0.613106
145826                 demagogicher    0.601221
41395                   demagogiche    0.600751
63269                     demagogie    0.598418
80315              demagogenconvent    0.593492
116580               geammtausdruck    0.590426
206209                 demagogichem    0.589134
167343                  majeimdruck    0.587386
121332                 leipzigdruck    0.587146
177741                  demagogenin    0.578579
94220                 demagogenthum    0.577759
93314                demagogenklubs    0.573550
29660                  demagogichen    0.573391
24744                     demagogen    0.568412
186486            straßendemokratie    0.563139
145991               deutchdemokrat    0.554925
83695                      demagoge    0.554044
63337                    demagogich    0.553544
162711           demoaritokratichen    0.552265
184060  demokratichpropaganditicher    0.552048
114452                    demkratie    0.550648
89205                geammteindruck    0.550191
198038       repräentativdemokratie    0.549877
187330                 demagogiches    0.547311
129285                 geitesdrucks    0.546737
215670             geammteindruckes    0.544283
157687             antidemokratiche    0.543279
105619             antidemagogichen    0.542331
162122              deklamatoricher    0.540049

Finding words similar to 'Revolutionspresse'...
Number of unique words: 269204
Most similar words for 'Revolutionspresse':
                          word  similarity
116468         revolutionspree    0.859073
47522       revolutionsveruche    0.854293
57216       revolutionsgräueln    0.851040
236063       revolutionsemiärs    0.850698
57628        revolutionsveruch    0.844562
211909       revolutionstracht    0.838932
241313      revolutionsaufrufe    0.832676
200186        revolutionskriis    0.829005
148957     revolutionsumtriebe    0.826244
37365          revolutionsgeit    0.824892
125548      revolutionszutande    0.822222
206667      revolutionsprincip    0.818134
117517     revolutionsveruchen    0.817804
234897        revolutionsplatz    0.816390
198418       revolutionsepoche    0.815463
202480       revolutionsplatze    0.815281
136225     revolutionszutandes    0.815264
178002       revolutionszwecke    0.814688
210601     revolutionsgedenken    0.812859
250817      revolutionsreultat    0.812319
177742  revolutionsverzweigung    0.810537
120023        revolutionspläne    0.809398
180601        revolutionsfeuer    0.803397
143545   revolutionsgechichten    0.802304
137647     revolutionsprediger    0.802187
154694         revolutionswege    0.799610
101088       revolutionstürmen    0.796306
203673          revolutionsgae    0.795443
249879       revolutionstaumel    0.793148
94690        revolutionsgefahr    0.792080
61647         revolutionswerke    0.791469
195742        revolutionseuche    0.791312
203379      revolutionsbankett    0.790931
138288         revolutionsturm    0.790232
151996        revolutionskugel    0.788463
147423   revolutionspropaganda    0.786549
37148     revolutionsgechichte    0.786503
137318         revolutionsgift    0.783503
166894           revolutionsge    0.782716
95628       revolutionsprinzip    0.782686

Finding words similar to 'radikale Presse'...
Number of unique words: 269204
Most similar words for 'radikale Presse':
                        word  similarity
254098         radikalrevolu    0.726243
21226               radikale    0.724742
239998        radikaldemokra    0.717985
63472   radikaldemokratichen    0.717490
245318   radikaldemokratiche    0.717396
141251           radikaldemo    0.711409
69293             radikalkur    0.708576
45311         radikalliberal    0.705080
21137                radikal    0.703495
238239  radicaldemokratichen    0.702906
208886         radikalreform    0.700832
21125      radikaldemokraten    0.699177
232267          radikaliiren    0.697627
53487              radikales    0.696675
14618              radikalen    0.695705
160753             radikalem    0.695509
68712              radikalis    0.694924
104149          radikaliemus    0.690657
66427       radikalreformern    0.688770
44902        radikalreformen    0.688229
130463          radikalismas    0.683632
154321  radikalprinzipiellen    0.681963
83535              radikalte    0.679950
73741               radicale    0.679712
45301        radikalreformer    0.679022
9545            radikalismus    0.675913
261316          radicalismus    0.674855
115744         radikalmittel    0.672972
139637         radikalformer    0.672828
259641         radikalquelle    0.672541
37148   revolutionsgechichte    0.671666
133606            radikalere    0.665801
152676           radikaleres    0.661600
236063     revolutionsemiärs    0.660882
56149             radikalten    0.660692
95612            radikaleren    0.655152
57216     revolutionsgräueln    0.654595
116468       revolutionspree    0.651747
200186      revolutionskriis    0.651511
244053   revolutionsblättern    0.651443

Finding words similar to 'ultraradikale Presse'...
Number of unique words: 269204
Most similar words for 'ultraradikale Presse':
                        word  similarity
69477          ultraradikale    0.764547
124866        ultraradikaler    0.763334
27631         ultraradikalen    0.761325
83511         ultraradikales    0.759280
151660        ultraradikalis    0.758912
126796          ultraradikal    0.756059
83486            ultraradika    0.755531
139812        ultraradikalem    0.746672
119267        ultraradicalen    0.737009
253597          ultrarepubli    0.728054
249932        ultrapreußiche    0.721201
110007             ultraradi    0.713439
107774     ultraradikalismus    0.704916
78265           ultrademokra    0.694854
258733         ultrapreußich    0.694277
184578      ultrareaktionäre    0.692743
195748          ultrapreußen    0.674979
196349         ultraprogreit    0.667439
66873           ultradäniche    0.660245
36691             ultradänen    0.650560
196337       ultraprogreiten    0.648772
202802            ultrablatt    0.647223
174415        ultratorytiche    0.645686
34472     ultrarevolutionäre    0.644412
137682          ultrablätter    0.641125
35167    ultrarevolutionären    0.638449
262048       ultratoryticher    0.637359
115049             ultratory    0.636231
77795   ultrarepublikanichen    0.636038
85396         ultrarevolutio    0.635221
149326         ultradänichen    0.634978
5843               ultradäne    0.634300
263831      ultraintrikanten    0.632869
89477     ultraprogreitichen    0.632357
143616             ultradäni    0.631765
20503            ultradeutch    0.631491
240792      ultrarepublikani    0.630591
90604       ultrapreußenthum    0.626905
230653           ultramodera    0.626067
120387     ultrademokratiche    0.625382

Finding words similar to 'jüdische Presse'...
Number of unique words: 269204
Most similar words for 'jüdische Presse':
                       word  similarity
53058              jüdichen    0.762947
104735    jüdichchritlichen    0.757819
24810               jüdiche    0.738486
246325     jüdichchritliche    0.738120
98192      preußichezeitung    0.704437
53082              jüdicher    0.696406
216559             judaskuß    0.689546
150805               jüdifj    0.678205
53043              jüdinnen    0.677250
154300            judaismus    0.666471
212709        völkerzeitung    0.656274
263560           judaskrieg    0.649187
227722     kirchenzeitungen    0.645865
251199       centralzeitung    0.643459
80814              jüdiches    0.641086
227744       kirchenzeitung    0.639055
32003           jurazeitung    0.638907
98928              giornale    0.638233
191902          aendzeitung    0.636431
255700       lokalzeitungen    0.636376
224079   schweizerzeitungen    0.635673
180370        zialzeitungen    0.633723
127902             jüdichem    0.631090
44178       nationalzeitung    0.629114
34046                gazeta    0.627195
252789     inrikestidningar    0.624786
58284           meelzeitung    0.623849
37864          journalismus    0.622543
124622          hebräischen    0.621399
187672               novine    0.621376
162273     literaturzeitung    0.620128
48796          hofzeitungen    0.619150
252828       ediktalzeitung    0.618130
198032  staatsbürgerzeitung    0.617177
229882           iarzeitung    0.614658
124646           hebräische    0.612252
135086        frauenzeitung    0.609955
166944         gazetapolska    0.608955
121882           bibraichen    0.608936
74950               közlöny    0.608856

Finding words similar to 'Judenpresse'...
Number of unique words: 269204
Most similar words for 'Judenpresse':
                             word  similarity
104735          jüdichchritlichen    0.702983
154300                  judaismus    0.701650
246325           jüdichchritliche    0.692386
216559                   judaskuß    0.683672
263560                 judaskrieg    0.664429
53058                    jüdichen    0.658590
24810                     jüdiche    0.657276
53043                    jüdinnen    0.653986
80555                 jesuitismus    0.644605
150805                     jüdifj    0.639566
53082                    jüdicher    0.624153
129285               geitesdrucks    0.618904
121332               leipzigdruck    0.616814
51663               preußenfeinde    0.616770
88054              volksausdrucks    0.609484
71523                  judaspreis    0.606305
98192            preußichezeitung    0.605557
32003                 jurazeitung    0.599080
234948                geitesdruck    0.596606
212709              völkerzeitung    0.594638
80814                    jüdiches    0.592981
2223                     judenbaß    0.586693
268553                   jesuiten    0.586452
127902                   jüdichem    0.583683
191902                aendzeitung    0.575932
223879          preußenfeindchaft    0.575639
2340                     judenhaß    0.573121
218589         judenemancipations    0.570143
63430           judenemancipation    0.569650
68069                  genzeitung    0.565392
176550                 judividuen    0.562904
66250                volkszeitung    0.561240
204610                      judia    0.561184
24461              kriegerzeitung    0.560028
203483  völkerunterdrückungsdient    0.557919
148560                julidynatie    0.557344
238581                preußenlied    0.556306
17415                      presse    0.553535
167343                majeimdruck    0.553354
25796                kuntausdruck    0.553325

Finding words similar to 'liberale Presse'...
Number of unique words: 269204
Most similar words for 'liberale Presse':
                           word  similarity
10884                  liberale    0.794709
32832                 liberalem    0.782196
22774                 liberales    0.768614
266386               liberalwäh    0.764385
1325                  liberalen    0.763204
12107                 liberaler    0.759082
228554              liberalessm    0.758706
8377                liberaleren    0.755860
266009                liberalsr    0.754135
191967           liberalgeinnte    0.753318
232950              liberalerer    0.751165
150295              liberaleres    0.746486
12409                   liberal    0.737932
12453         liberalpapitichen    0.737915
72395               liberalerem    0.735299
149900                liberalis    0.734558
16663         liberalkonervatis    0.733910
33967                liberalten    0.725705
22841                liberalere    0.725086
215353         liberalencantone    0.723622
55620      liberalkonervatismus    0.722083
35651          freidemokratiche    0.719598
248741           gratisjournale    0.706286
14735              liberalismus    0.703691
85552                 liberalte    0.702960
191163            libertinismus    0.689983
184720      liberalkonervativen    0.683718
53208                liberalter    0.683529
83908   liberalkontitutionellen    0.679828
74733          freievangelichen    0.675409
261806     freiheitsmörderichen    0.672333
114808       liberalkonervative    0.669340
67866          freiheitsmännern    0.668031
249673         freiheitschreier    0.666972
167599           freiheitsarmee    0.666724
254476          freiheitsfeinde    0.665937
252977        freichaarenpartei    0.665177
162273         literaturzeitung    0.664888
162909      volksfreiheitlichen    0.664239
134201          freiheitsmörder    0.663974

Finding words similar to 'Freimaurerpresse'...
Number of unique words: 269204
Most similar words for 'Freimaurerpresse':
                       word  similarity
137645        kupferdrucker    0.662102
122663        prachtabdruck    0.661236
217200         steindrucker    0.656658
121332         leipzigdruck    0.653978
167343          majeimdruck    0.644456
204035       landturmpreern    0.637735
106462        ausdrucksweie    0.628757
76068          steuerdrucke    0.626204
229341     rathsbuchdrucker    0.613064
193137  chwarzaufgedruckten    0.612607
189729        freimaurerkon    0.609779
189798    freimaurerkongreß    0.608951
116580       geammtausdruck    0.604397
89832          gewiensdruck    0.604076
179460          steuerdruck    0.603712
245616     kupferdruckvelin    0.602247
223661    nachdrucksprocees    0.600809
174940      buchdruckerpree    0.598768
43029            freimaurer    0.597832
234948          geitesdruck    0.595553
148174         ausdruckweie    0.595441
129285         geitesdrucks    0.589890
234645         plänkelfeuer    0.588977
263778      kammerdruckerei    0.588924
98679        beamtetendruck    0.586740
226255          stampfmühle    0.586131
194543           steindruck    0.584247
211363            perldruck    0.581634
251391          kupferdruck    0.580478
129287         preußenfreer    0.573880
198765         preßklaverei    0.571802
219755       präcluivtermin    0.571520
202032           segeldruck    0.571106
228770             zeidruck    0.569048
218367        preßstrafytem    0.568515
91987     brochürenchreiber    0.567639
217440      präventivmittel    0.563556
156629        umfaungsmauer    0.561877
253764      pferdegetrappel    0.559948
118306     druckvelinpapier    0.559615

Finding words similar to 'Logenpresse'...
Number of unique words: 269204
Most similar words for 'Logenpresse':
                        word  similarity
121332          leipzigdruck    0.617513
223661     nachdrucksprocees    0.592748
76068           steuerdrucke    0.586299
84169            nachdrucker    0.579951
229341      rathsbuchdrucker    0.577002
182063  journalunterdrückung    0.574464
266329        nachdruckamere    0.571615
263778       kammerdruckerei    0.570618
125755           druckenlaen    0.569806
137645         kupferdrucker    0.567106
106462         ausdrucksweie    0.566199
223474      chrifttellernden    0.564136
23691            druckchrift    0.559223
17415                 presse    0.558868
46769             lobpreiern    0.553583
179460           steuerdruck    0.547694
268233           tipografica    0.547671
91987      brochürenchreiber    0.547541
189618       schrifttellerko    0.547152
206987           printzinger    0.546325
129285          geitesdrucks    0.545533
100262       schrifttellerin    0.545214
125991         schriftchauer    0.544781
221720        verlagshandlun    0.544460
193618       schrifttellerei    0.544182
126811            verlagsans    0.541613
168098                 loger    0.541375
265046          nachdruckame    0.539967
59682       schrifttellerver    0.539529
115470          nachdruckamt    0.538431
232558                 press    0.538041
23701          druckchriften    0.537531
234948           geitesdruck    0.537488
223475         chrifttellern    0.535719
31807       hofbuchdruckerei    0.535377
2909           schriftteller    0.534297
131064           schriftzuge    0.534125
265062     winkeldruckereien    0.533721
156785            logenreihe    0.533721
221934          journalpreen    0.531640

Finding words similar to 'Pressunfug'...
Number of unique words: 269204
Most similar words for 'Pressunfug':
                       word  similarity
17415                presse    0.714191
234017       publikaniirung    0.703498
128987          publikanern    0.683185
47800          publikaniche    0.673279
101830        publikanismus    0.669950
232558                press    0.653942
222143        publikanichem    0.651112
120929        publikanicher    0.647032
174890          publicirung    0.645552
66354         publicirenden    0.644861
26664         publikanichen    0.641559
76566            publikaner    0.639040
67197           publikanich    0.638315
20726           publizirung    0.637718
170644          pnblication    0.634125
67317          publizitiche    0.633566
74950               közlöny    0.631694
191733      propagandenween    0.631682
266857              pressen    0.628909
182842          publizirter    0.624734
8338       veröffentlichung    0.622957
7002             publicität    0.621550
31816            publiciren    0.621144
110893        publizitiches    0.619076
7145             propaganda    0.618993
171678  publicationspatente    0.617624
79138            publikaten    0.617461
100131        publizitichen    0.616811
96572      publikanerhaufen    0.616046
25259            publizität    0.615789
151863           publicirte    0.615300
107718      veröffentlichun    0.613885
83368        propagandismus    0.613883
59499           publikaners    0.612082
72405           publication    0.611721
649                ausdruck    0.609963
21422           publikation    0.607981
232867      eröffentlichung    0.606373
158301          propagandis    0.605391
7664             publiziten    0.601917

Finding words similar to 'Wühlerpresse'...
Number of unique words: 269204
Most similar words for 'Wühlerpresse':
                     word  similarity
137645      kupferdrucker    0.758320
106462      ausdrucksweie    0.703728
43867         drückendter    0.694521
84169         nachdrucker    0.689218
48893        hülfsprieter    0.676714
199164       struwelpeter    0.676278
89832        gewiensdruck    0.675221
245616   kupferdruckvelin    0.675098
121332       leipzigdruck    0.674587
179460        steuerdruck    0.673340
49541        stempeltaren    0.672227
223661  nachdrucksprocees    0.670938
17415              presse    0.663422
57722      präentirteller    0.662035
251391        kupferdruck    0.661854
70091         steuerruder    0.656132
234948        geitesdruck    0.651370
116580     geammtausdruck    0.651193
149490       stempelteuer    0.650791
153652        stempeltare    0.648836
232558              press    0.643780
266857            pressen    0.640127
148174       ausdruckweie    0.640100
98679      beamtetendruck    0.639698
210424           drückter    0.637237
229341   rathsbuchdrucker    0.634992
76068        steuerdrucke    0.634314
217440    präventivmittel    0.634051
156842           pression    0.631223
36284       nungsausdruck    0.629811
78689          widerdruck    0.628500
92133       hülfsprediger    0.626410
57241         gefällsüber    0.624234
228734     ausdruckvollte    0.623501
201683        preßmachine    0.621690
147169         gegendruck    0.621517
167343        majeimdruck    0.620350
215923     preßverbrecher    0.618582
89750      gehülfsprieter    0.618166
33067             abdruck    0.617544

Finding words similar to 'Gegenpresse'...
Number of unique words: 269204
Most similar words for 'Gegenpresse':
                     word  similarity
64816        gegendruckes    0.852296
147169         gegendruck    0.848778
78689          widerdruck    0.754183
58801          tegenspold    0.715998
221715          gegencoup    0.691163
33067             abdruck    0.690666
38818          gegenchlag    0.685655
122859       entgegenpuft    0.684093
3608            nachdruck    0.679945
188527         gegentreue    0.676972
139406        gegenchrift    0.676804
224490         gegenkraft    0.675702
108869        gegenrütung    0.673324
261625      wiederabdruck    0.673180
46864      gegenpiegelung    0.671826
43537           gegentück    0.671361
245215          gegenrech    0.671239
240337       gegenbechluß    0.670550
93492         gegenbeuche    0.670290
111082        gegenchlage    0.670189
115781     gegenvorchlage    0.669613
72078     gegenvortellung    0.669412
14106        gegenprotets    0.668108
258294        gegenprotet    0.665949
23204      gegenbemerkung    0.665108
212190      gegendemontra    0.663687
17415              presse    0.662695
101381           gegenpar    0.661813
92510            gegenübr    0.660766
261356          reindruck    0.659019
124047         antipreußi    0.658793
156842           pression    0.658585
193448      gegenchreiber    0.658283
268773          rpression    0.656902
214983           gegenaze    0.656291
223661  nachdrucksprocees    0.654021
22967           gegenwehr    0.652705
237279         gegenlaufe    0.652050
102064         gegenzeich    0.651985
164812        antirepubli    0.651194

Nun da wir Listen mit verschiedenen ähnlichen Worten haben müssen wir diese Listen in eine große Liste kombinieren. Am besten machen wir dies basierend auf dem Similarity score, der auf einer Skala von 1 (mehr oder weniger gleich) bis -1 (könnten nicht unterschiedlicher sein) die Ähnlichkeit des ursprünglichen Suchbegriffs und des gefundenen ähnlichen Wortes bestimmt.
Dabei sollte der Similarity score von Lügenpresse niedriger sein als der der anderen Synonyme. Arbiträr, basierend auf den ergebnissen würde ich den Similarity score für Lügenpresse auf 0.58 und für die anderen Begriffe auf mindestens 0.63 setzen. Da bei einem niedrigeren Score schon Begriffe wie händedruck (0.629466 zu Hetzpresse) eingeschlossen wären aber lügenboten beispielsweise einen Score von 0.581949 zu Lügenpresse hat.


In [ ]:
synonyme_2 = {
    "lügentrafens",
    "lügentrafen",
    "lügenfluth",
    "lügenpot",
    "lügengechichte",
    "lügenkaiers",
    "leipzigdruck",
    "lügenkünte",
    "lügentoff",
    "lügengeit",
    "lügenytem",
    "lügenmaske",
    "lügenteige",
    "lügenberichten",
    "lügengepinnt",
    "lügenberichte",
    "lügentaktik",
    "presse",
    "lügenboten",
    "lügenfabrik"

    "lügen", "lüge", "lie", "lies", "lügengeit", "lagau", "liar", "mentirt",
    "gelogen", "mentirte", "lügenhaftete", "mentiren", "lügenytem", "lieg",
    "lied", "liehe", "lügenhaften", "lügenhafte", "lügenteige", "lügenhaft",
    "fausse", "liee", "lügt", "fausses", "liege", "lapsus", "liegenheit",
    "gelobe", "lügenpot", "lieh", "liari", "liarde", "liel", "lügenfluth",
    "lieds", "gelob", "liegenheiten", "falsos", "lügentoff", "lügentrafen",

    "erreur", "feil", "error", "feilbietung", "feldauftellung", "feilbot",
    "fehlerhaf", "rechenfehler", "rechnungsfehler", "terlaungsfehler",
    "lügenberichten", "druckfehlerberichtigung", "uebereilungsfehler",
    "feilgeboten", "rangtitel", "fehlchluß", "formfehler", "lügenmaske",
    "lügenberichte", "feilzubietenden", "fehlerhaftes", "feilheit",
    "feilbieten",

    "lügenblätter", "lügengechichte", "jeuitenblatt", "feigenblatt",
    "erbauungsblatt", "gierungsblatt", "lügentrafens", "lügentoff",
    "tungsblatt", "lügenytem", "lügengeit", "regblatt", "jeuitenblatte",
    "oppoitionsblatt", "verordnblatt", "flugblatt", "llandbuch",
    "lügenkünte", "lügenteige", "börenblatt", "lebrbuch", "lügenboten",
    "lügenkaiers", "blaebälgen", "lügengewebe", "lappland",
    "feigenblatts", "ragierungsblattes", "späheblatt", "verlagsbuchh",
    "leebuch", "kleeblatt", "pondenzblattes", "tagblatt", "volksblatt",
    "mailänderblatt",

    "herzblatt", "zelblatt", "geetzblatt", "hatzfeld", "holzdiebtahl",
    "hatkett", "zahlbrett", "hatzfeldt", "haspthebel", "witzblatt",

    "geitesdruck", "geitesdrucks", "abdruck", "nachdruck",
    "nachdrucksprocees", "leipzigdruck", "ausdruck", "händedrucks",
    "presse", "pressen", "drückendten", "pression", "press",
    "unterdrucken",

    "schmeißblatt", "scherzblatt", "stichblatt", "bundesblatt",
    "adreßblatt", "winckelblech", "schlußheft", "oppoitionsblatt",
    "börenblatt", "haltungsblatt", "mailänderblatt", "feigenblatt",
    "kreisblatt", "rungsblatt", "hauptblatt",

    "kupferdrucker", "stempeltaren", "stempeltare", "preßmachine",
    "ausdrucksweie", "stempelteuer", "präentirteller", "drückendter",
    "nachdrucker",

    "pamphletiten", "pamphletit", "pamphlete", "pamphleten", "pamphlet",
    "pamphletchreiber", "pamphlets", "pamphle",

    "parteinachricht", "parteipolemik", "parteiblatte", "parteiblatt",
    "parteireibereien", "parteiabicht", "parteiüchti", "parteiauflöung",
    "parteifetzug", "parteifraktion", "parteibechluß", "parteiumtriebe",
    "parteibaß", "parteiblätter", "parteigetreibe", "parteihäuptling",
    "partianenkorps", "parteiblät",

    "demagogendespotie", "demagogenriecherei", "demagogenkünten",
    "demagogenkünte", "demagogichdemokratichen", "demagogenriechern",
    "demagogenwirthchaft", "demagogenunteruchungen",

    "revolutionspree", "revolutionsveruche", "revolutionsgräueln",
    "revolutionsemiärs", "revolutionsveruch", "revolutionstracht",
    "revolutionsaufrufe", "revolutionskriis", "revolutionsumtriebe",
    "revolutionsgeit", "revolutionszutande", "revolutionsprincip",
    "revolutionsveruchen", "revolutionsplatz", "revolutionsepoche",
    "revolutionsplatze", "revolutionszutandes", "revolutionszwecke",
    "revolutionsgedenken", "revolutionsreultat", "revolutionsverzweigung",
    "revolutionspläne", "revolutionsfeuer", "revolutionsgechichten",
    "revolutionsprediger", "revolutionswege", "revolutionstürmen",
    "revolutionsgae", "revolutionstaumel", "revolutionsgefahr",
    "revolutionswerke", "revolutionseuche", "revolutionsbankett",
    "revolutionsturm", "revolutionskugel", "revolutionspropaganda",
    "revolutionsgechichte", "revolutionsgift", "revolutionsge",
    "revolutionsprinzip",

    "radikalrevolu", "radikale", "radikaldemokra",
    "radikaldemokratichen", "radikaldemokratiche", "radikaldemo",
    "radikalkur", "radikalliberal", "radikal", "radicaldemokratichen",
    "radikalreform", "radikaldemokraten", "radikaliiren", "radikales",
    "radikalen", "radikalem", "radikalis", "radikaliemus",
    "radikalreformern", "radikalreformen",

    "ultraradikale", "ultraradikaler", "ultraradikalen",
    "ultraradikales", "ultraradikalis", "ultraradikal", "ultraradika",
    "ultraradikalem", "ultraradicalen",

    "jüdichen", "jüdichchritlichen", "jüdiche", "jüdichchritliche",
    "preußichezeitung", "jüdicher", "judaskuß", "jüdifj", "jüdinnen",

    "judaismus", "judaskrieg", "jesuitismus",

    "liberale", "liberalem", "liberales", "liberalwäh", "liberalen",
    "liberaler", "liberalessm", "liberaleren", "liberalsr",
    "liberalgeinnte", "liberalerer", "liberaleres", "liberal",
    "liberalpapitichen", "liberalerem", "liberalis", "liberalkonervatis",
    "liberalten",

    "gegendruckes", "gegendruck", "widerdruck", "tegenspold",
    "gegencoup", "gegenchlag", "entgegenpuft", "gegentreue",
    "gegenchrift", "gegenkraft"
}

Jetzt da wir mehr oder weniger eine neue Synonyme Liste (oder Menge) haben können wir alles was wir mit der vorherigen Synonyme Liste gemacht haben auch hier noch mal mit der neune Liste (Synonyme 2.0) machen.

In [ ]:
pattern = '|'.join(synonyme_2)

df_test = df[df['plainpagefulltext'].str.contains(pattern, na=False)]
print(len(df_test))

7601 Seiten mit Treffern sind bei einem Datensatz von 7666 Seiten fast jede Seite. Die Frage ist wie Sinnvoll ein paar dieser "Synonyme" wirklich sind. Die Tatsache dass über 99% meines Datensatzes in meine Suchkriterien passt sehe ich als klares Indiz dafür dass meine Suchkriterien sind. Besonders Begriffe wie jüdischer sind zwar semantisch ähnlich zu Judenpresse, aber das macht sie noch lange nicht zu synonymen von Lügenpresse. Auch andere Wörter wie Lied, sind zwar vielleicht semantisch ähnlich zu einem der Synonyme. Das macht diese Wörter aber noch lange nicht relevant zur momentanen Analyse.
Daher habe ich beschlossen den Similarity score auf 0.77 hochzustellen. (Lied hat einen Similarity score von 0.76..)

In [ ]:
synonyme_3 = {
    "lügentrafens",
    "lügentrafen",
    "lügenfluth",
    "lügenpot",
    "lügengechichte",
    "lügenkaiers",
    "leipzigdruck",
    "lügenkünte",
    "lügentoff",
    "lügengeit",
    "lügenytem",
    "lügenmaske",
    "lügenteige",
    "lügenberichten",
    "lügengepinnt",
    "lügenberichte",
    "lügentaktik",
    "lügen", "lüge", "lie", "lies", "lügengeit", "lagau", "liar", "mentirt",
    "gelogen", "mentirte", "lügenhaftete", "mentiren", "lügenytem",
    "schmeißblatt",
    "revolutionspree", "revolutionsveruche", "revolutionsgräueln", "revolutionsemiärs",
    "revolutionsveruch", "revolutionstracht", "revolutionsaufrufe", "revolutionskriis",
    "revolutionsumtriebe", "revolutionsgeit", "revolutionszutande", "revolutionsprincip",
    "revolutionsveruchen", "revolutionsplatz", "revolutionsepoche", "revolutionsplatze",
    "revolutionszutandes", "revolutionszwecke", "revolutionsgedenken", "revolutionsreultat",
    "revolutionsverzweigung", "revolutionspläne", "revolutionsfeuer", "revolutionsgechichten",
    "revolutionsprediger", "revolutionswege", "revolutionstürmen", "revolutionsgae",
    "revolutionstaumel", "revolutionsgefahr", "revolutionswerke", "revolutionseuche",
    "revolutionsbankett", "revolutionsturm", "revolutionskugel", "revolutionspropaganda",
    "revolutionsgechichte", "revolutionsgift", "revolutionsge", "revolutionsprinzip",
    "liberale", "liberalem",
    "gegendruckes", "gegendruck"

}

pattern = '|'.join(synonyme_3)

df_test = df[df['plainpagefulltext'].str.contains(pattern, na=False)]
print(len(df_test))

Selbst ein Hochschrauben des Similarity scores auf 0.85 hilft nicht wirklich

In [ ]:
synonyme_4 = {
    "lügentrafens",
    "lügentrafen",
    "lügenfluth",
    "lügenpot",
    "lügengechichte",
    "lügenkaiers",
    "leipzigdruck",
    "lügenkünte",
    "lügentoff",
    "lügengeit",
    "lügenytem",
    "lügenmaske",
    "lügenteige",
    "lügenberichten",
    "lügengepinnt",
    "lügenberichte",
    "lügentaktik",
    "lügen",
    "lüge",
    "lie",
    "revolutionspree",
    "revolutionsveruche",
    "revolutionsgräueln",
    "revolutionsemiärs"

}

pattern = '|'.join(synonyme_4)

df_test = df[df['plainpagefulltext'].str.contains(pattern, na=False)]
print(len(df_test))

Ein letzter Versuch nur mit den Wörtern die ähnlich zu Lügenpresse sind.

In [ ]:
synonyme_5 = {
    "lügentrafens",
    "lügentrafen",
    "lügenfluth",
    "lügenpot",
    "lügengechichte",
    "lügenkaiers",
    "leipzigdruck",
    "lügenkünte",
    "lügentoff",
    "lügengeit",
    "lügenytem",
    "lügenmaske",
    "lügenteige",
    "lügenberichten",
    "lügengepinnt",
    "lügenberichte",
    "lügentaktik",
}

pattern = '|'.join(synonyme_5)

df_test = df[df['plainpagefulltext'].str.contains(pattern, na=False)]
print(len(df_test))

Und direkt sind es wieder 0 Ergebnisse. Aller Wahrscheinlichkeit nach sind auch hier wieder ein Großteil der Ergebnisse auf sehr allgemeine Wörter wie Lüge zurück zu führen.

In [ ]:
synonyme_6 = {
    "lügen",
    "lüge"
}

pattern = '|'.join(synonyme_3)

df_test = df[df['plainpagefulltext'].str.contains(pattern, na=False)]
print(len(df_test))

Um nicht zu sagen alle.

Ich weiß ehrlich geasagt nicht ob es Sinn macht diesen Weg weiter zu verfolgen und würde deswegen hier erst mal pausieren. Ob es sinnvoll ist die similar word analyse weiter zu verfolgen und wenn ja wie kann ich in meinem nächsten Treffen mit Dr. Oberbichler besprechen.
Ansonsten würde ich jetzt noch eine Sentiment-analyse der Synonym 1.0 Ergebnisse starten, dannach meine Analyse aber erst einmal für beendet erklären.